# Leukemia scRNA-seq — Condition-Specific Programs After Transplant

Reproduces **Section 2.4** of the bcNMF paper.

**Setup:** Single-cell RNA-seq from bone marrow mononuclear cells (BMMCs) of a leukemia patient before and after stem cell transplantation (patient 27, Zheng et al. 2017). Healthy donor BMMCs serve as the background to remove baseline bone marrow variation.

## Data

- **Patient 27 pre/post-transplant BMMCs** — 10x Genomics (Zheng et al., *Nat. Commun.* 2017).
  - Download: https://www.10xgenomics.com/datasets (search "fresh 68k PBMCs" or the companion bone-marrow dataset)
- **Healthy donor BMMCs** — same publication / 10x Genomics portal.
- Place raw count matrices under `data/leukemia/` (not tracked by git).

After standard QC: retain 18,976 protein-coding genes, select top 3,000 HVGs.

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import umap

import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from bcNMF import contrastive_nmf_poisson, nmf_poisson

## 1. Load, QC, and select HVGs

In [ ]:
# TODO: load AnnData objects for pre-transplant, post-transplant, and healthy donor
# Apply QC filters, restrict to protein-coding genes, select top 3,000 HVGs
#
# X_target    : np.ndarray (3000, n_patient_cells)   -- pre + post transplant cells
# X_background: np.ndarray (3000, n_healthy_cells)   -- healthy donor cells
# condition   : array of 'pre' / 'post' labels for X_target (evaluation only)
raise NotImplementedError('Add data loading code here')

## 2. Fit NMF and bcNMF (K = 10, Poisson)

In [ ]:
K = 10
alpha = 1.0  # tune via grid search

W_nmf, H_nmf, _ = nmf_poisson(X_target, K=K, niter=300)
W_bc, H_X_bc, H_Y_bc, _ = contrastive_nmf_poisson(X_target, X_background, K=K, alpha=alpha, niter=300)

## 3. UMAP visualisation and ARI (Fig. 4a–c)

In [ ]:
reducer = umap.UMAP(random_state=42)

emb_nmf = reducer.fit_transform(H_nmf.T)
emb_bc  = reducer.fit_transform(H_X_bc.T)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, emb, title in zip(axes, [emb_nmf, emb_bc], ['NMF', 'bcNMF']):
    ax.scatter(emb[:, 0], emb[:, 1], c=(condition == 'post').astype(int), cmap='coolwarm', s=2)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 4. Top gene loadings per condition-associated factor (Fig. 4d–e)

Regress transplant status on each factor's usage to identify the most condition-informative topics. For the top factors, display top-5 genes and per-gene Welch t-test statistics.

In [ ]:
from scipy import stats
from sklearn.linear_model import LogisticRegression

# TODO: regress condition on H_X_bc.T, rank factors by p-value,
# then extract top genes from W_bc for the most significant factors
pass